In [ ]:
!pip install 'accelerate>=0.26.0'
!pip install datasets
!pip install transformers
!pip install torch
!pip install pillow
!pip install timm

In [ ]:
from datasets import load_dataset

# load the Yelp dataset (full review dataset with train and test splits)
dataset = load_dataset("yelp_polarity")

# inspect the dataset to understand the structure
print(dataset)

In [ ]:
# Access the train split
train_dataset = dataset['train']

# Print the first example
print(train_dataset[0])

In [ ]:
# select the "train" and "test" splits
train_dataset = dataset["train"]
test_dataset = dataset["test"]

# filter for restaurant-related reviews in the train and test datasets
restaurant_train_reviews = train_dataset.filter(
    lambda x: "restaurant" in x["text"].lower()
)

restaurant_test_reviews = test_dataset.filter(
    lambda x: "restaurant" in x["text"].lower()
)

number_of_reviews = 5000
subset_train_reviews = restaurant_train_reviews.shuffle(seed=42).select(range(number_of_reviews))
subset_test_reviews = restaurant_test_reviews.shuffle(seed=42).select(range(number_of_reviews))

# create a DatasetDict to return both train and test datasets
subset_dataset = {
    "train": subset_train_reviews,
    "test": subset_test_reviews
}

# display the structure to match the requested format
from datasets import DatasetDict
yelp_restaurant_dataset = DatasetDict(subset_dataset)

# print the dataset structure
print(yelp_restaurant_dataset)

In [ ]:
# Access the train split
yelp_restaurant_dataset['train'][0]

In [ ]:
from transformers import AutoTokenizer

# load a pre-trained model and tokenizer - DistilBERT for sentiment classification
model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# tokenize the dataset
def tokenize_function(examples):
    return tokenizer(examples["text"], 
                     padding = "max_length", 
                     truncation = True, 
                     max_length = 512)

# apply the tokenization function to the entire dataset
tokenized_datasets = yelp_restaurant_dataset.map(tokenize_function, batched=True)
tokenized_datasets

In [ ]:
from transformers import AutoModelForSequenceClassification                        
import torch

# load pre-trained model for sequence classification
model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, 
                                                           num_labels = 2)

# determine the device
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# move the model to the selected device
model.to(device)

In [ ]:
from transformers import Trainer, TrainingArguments

# set up training arguments
training_args = TrainingArguments(
    output_dir = "./results",                  # Directory to save results
    eval_strategy = "epoch",                   # Evaluate model after each epoch
    save_strategy = "epoch",                   # Save the model after each epoch
    learning_rate = 2e-5,                      # Learning rate
    per_device_train_batch_size = 16,          # Batch size for training
    per_device_eval_batch_size = 16,           # Batch size for evaluation
    num_train_epochs = 3,                      # Number of training epochs
    weight_decay = 0.01,                       # Weight decay for regularization
    logging_dir = "./logs",                    # Directory for logs
    logging_steps = 10,                        # Log every 10 steps
    save_steps = 500,                          # Save the model every 500 steps
    load_best_model_at_end = True,             # Load the best model at the end of training
)

# set up the Trainer
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = tokenized_datasets["train"],
    eval_dataset = tokenized_datasets["test"],
)

# fine-tune the model
trainer.train()

In [ ]:
# save the fine-tuned model and tokenizer
model.save_pretrained("./results/final_model")
tokenizer.save_pretrained("./results/final_tokenizer")

In [ ]:
# evaluate the model
eval_results = trainer.evaluate()
print(f"Evaluation results: {eval_results}")

In [ ]:
from transformers import AutoTokenizer, \
                  AutoModelForSequenceClassification
import torch

# reload the model and tokenizer
new_model = AutoModelForSequenceClassification.from_pretrained("./results/final_model")
new_tokenizer = AutoTokenizer.from_pretrained("./results/final_tokenizer")

# determine the device
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
new_model.to(device)

# good review
sentence = '''
I had an amazing experience dining at this restaurant last night. From the moment 
we walked in, the staff made us feel welcomed and were incredibly attentive. Our 
server was friendly, knowledgeable, and made great recommendations from the menu.
The food was absolutely delicious. I had the grilled salmon, and it was cooked to 
perfection—tender, flavorful, and served with a lovely citrus glaze that 
complemented it beautifully. The roasted vegetables on the side were fresh and 
perfectly seasoned. My partner had the pasta, which was creamy and rich in flavor, 
with just the right amount of spice.
The ambiance was warm and inviting, with cozy lighting and tasteful decor. It was 
the perfect place to relax and enjoy a nice meal. The dessert, a decadent 
chocolate lava cake, was the perfect way to end the meal.
Overall, this restaurant exceeded my expectations in every way. Excellent food, 
exceptional service, and a wonderful atmosphere. I’ll definitely be back and 
highly recommend it to anyone looking for a great dining experience.
'''

# bad review
sentence = '''
I visited this place last night with high expectations after hearing some good 
things, but it was honestly one of the worst dining experiences I’ve had in a 
while. The service was incredibly slow, even though the restaurant wasn’t 
crowded. Our waiter seemed disinterested and forgot half of our order.
When the food finally came, it was cold and tasted bland. The pasta was 
overcooked, and my steak was underseasoned and chewy. The side of vegetables 
looked like they had been reheated from a previous meal.
To make things worse, the ambiance was far too noisy, and we had to wait an 
extra 20 minutes for the check. I tried to address my concerns with the manager, 
but they seemed uninterested in hearing feedback. Overall, I felt like I had 
wasted both my time and money.
I will definitely not be coming back, and I would not recommend this place to anyone.
'''

# tokenize the input sentence
inputs = new_tokenizer(sentence,
                       return_tensors = "pt", 
                       padding = True, 
                       truncation = True, 
                       max_length = 512)

# move inputs to MPS
inputs = {key: value.to(device) for key, value in inputs.items()}

# Put the model in evaluation mode
new_model.eval()  

# Perform inference
with torch.no_grad():
    # run the model to get predictions
    outputs = new_model(**inputs)
    
    # get the logits (raw scores) from the model output
    logits = outputs.logits
    
    # convert logits to probabilities using softmax
    probabilities = torch.nn.functional.softmax(logits, dim=-1)
    
    # get the predicted class (index of the maximum probability)
    predicted_class = torch.argmax(probabilities, dim=-1).item()
    
    # output the predicted sentiment
    if predicted_class == 1:
        print(f"Sentiment: Positive (Confidence: \
        {probabilities[0][1].item():.2f})")
    else:
        print(f"Sentiment: Negative (Confidence: \
        {probabilities[0][0].item():.2f})")

In [ ]:
from datasets import load_dataset

# load Yelp Reviews dataset with ratings from 1 to 5
dataset = load_dataset("yelp_review_full")

# display the structure of the dataset
print(dataset)

# select the "train" and "test" splits
train_dataset = dataset["train"]
test_dataset = dataset["test"]

# filter for restaurant-related reviews in the train and test datasets
restaurant_train_reviews = train_dataset.filter(
    lambda x: "restaurant" in x["text"].lower()
)

restaurant_test_reviews = test_dataset.filter(
    lambda x: "restaurant" in x["text"].lower()
)

# only use 5000 reviews for training
number_of_reviews = 5000
subset_train_reviews = restaurant_train_reviews.shuffle(seed=42).select(range(number_of_reviews))
subset_test_reviews = restaurant_test_reviews.shuffle(seed=42).select(range(number_of_reviews))

# create a DatasetDict to return both train and test datasets
subset_dataset = {
    "train": subset_train_reviews,
    "test": subset_test_reviews
}

# display the structure to match the requested format
from datasets import DatasetDict
yelp_restaurant_dataset = DatasetDict(subset_dataset)

# Print the dataset structure
print(yelp_restaurant_dataset)

In [ ]:
yelp_restaurant_dataset['train'][0]

In [ ]:
from transformers import AutoTokenizer

# load a pre-trained model and tokenizer - DistilBERT for 
# sentiment classification
model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# tokenize the data
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

# apply tokenization to the train and test sets
tokenized_datasets = yelp_restaurant_dataset.map(tokenize_function, batched=True)

In [ ]:
from transformers import AutoModelForSequenceClassification                        
import torch

model = AutoModelForSequenceClassification.from_pretrained(  
            model_checkpoint, num_labels = 5)

if torch.backends.mps.is_available():                        
    device = torch.device("mps")
else:
    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

In [ ]:
from transformers import Trainer, TrainingArguments

# set up training arguments
training_args = TrainingArguments(
    output_dir = "./results",                  # Directory to save results
    eval_strategy = "epoch",                   # Evaluate model after each epoch
    save_strategy = "epoch",                   # Save the model after each epoch
    learning_rate = 2e-5,                      # Learning rate
    per_device_train_batch_size = 16,          # Batch size for training
    per_device_eval_batch_size = 16,           # Batch size for evaluation
    num_train_epochs = 3,                      # Number of training epochs
    weight_decay = 0.01,                       # Weight decay for regularization
    logging_dir = "./logs",                    # Directory for logs
    logging_steps = 10,                        # Log every 10 steps
    save_steps = 500,                          # Save the model every 500 steps
    load_best_model_at_end = True,             # Load the best model at the end of training
)

# set up the Trainer
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = tokenized_datasets["train"],
    eval_dataset = tokenized_datasets["test"],
)

# fine-tune the model
trainer.train()

In [ ]:
# save the fine-tuned model and tokenizer
model.save_pretrained("./results/final_model_multiclass")
tokenizer.save_pretrained("./results/final_tokenizer_multiclass")

In [ ]:
# Evaluate the model on the test set
eval_results = trainer.evaluate()

# Print the evaluation results
print(eval_results)

In [ ]:
from transformers import AutoModelForSequenceClassification
from transformers import AutoTokenizer
import torch 

new_reviews = [
    "The food was amazing and the service was excellent!",
    "The restaurant was dirty and the food was cold.",
    "Decent experience, but nothing special."
]

# determine the device
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# load the tokenizer
new_tokenizer = AutoTokenizer.from_pretrained("./results/final_tokenizer_multiclass")

# tokenize the reviews
inputs = new_tokenizer(new_reviews,
                       padding = "max_length", 
                       truncation = True, 
                       return_tensors = "pt")

# move inputs to GPU / MPS
inputs = {key: value.to(device) for key, value in inputs.items()}

# load the fine-tuned model
new_model = AutoModelForSequenceClassification.from_pretrained(
    "./results/final_model_multiclass")
new_model.to(device)

In [ ]:
# Put the model in evaluation mode
new_model.eval()

# perform inference
with torch.no_grad():
    outputs = new_model(**inputs)
logits = outputs.logits
predictions = torch.argmax(logits, dim=-1)

In [ ]:
# map indices to star ratings
star_ratings = predictions + 1  # Assuming classes are 0-4, map to 1-5
for review, rating in zip(new_reviews, star_ratings):
    print(f"Review: {review}\nPredicted Star Rating: \
          {rating.item()}\n")

In [ ]:
from PIL import Image, ImageDraw
import requests

url = 'https://images.unsplash.com/' + \
      'photo-1563460716037-460a3ad24ba9'                     #A
if url.startswith('http'):                                   #B
    image = Image.open(requests.get(url, stream=True).raw)
else:    
    image = Image.open(url)                                  #C
image

#A Url of the image
#B if the image is from the web
#C the image is local 

In [ ]:
from transformers import DetrImageProcessor, DetrForObjectDetection
import torch

image_processor = DetrImageProcessor.from_pretrained("facebook/detr-resnet-50")
model = DetrForObjectDetection.from_pretrained("facebook/detr-resnet-50")
# model.config.id2label

inputs = image_processor(images = image,                   
                         return_tensors = "pt")
model.eval()

with torch.no_grad():
    outputs = model(**inputs)
    
target_sizes = torch.tensor([image.size[::-1]])
results = image_processor.post_process_object_detection(   
              outputs,
              target_sizes = target_sizes, 
              threshold = 0.9)[0]
print(results)

In [ ]:
draw = ImageDraw.Draw(image)

for score, label, box in zip(results["scores"], results["labels"], results["boxes"]):    
    print(                                                   
        f"Detected {model.config.id2label[label.item()]} with confidence "       
        f"{(score.item() * 100):.2f}% at {box}"        
    )
    box = [round(i, 2) for i in box.tolist()]                
    draw.rectangle(box, 
                   outline = 'green', 
                   width = 10)    
    
    draw.text((box[0], box[1]-10),                           
              model.config.id2label[label.item()], 
              fill = 'green')    
display(image)

In [ ]:
import torch
import requests
from PIL import Image

url = 'https://images.unsplash.com/' + \
      'photo-1491604612772-6853927639ef' #A Url of the image

# url = "https://upload.wikimedia.org/wikipedia/commons/thumb/3/3f/Walking_tiger_female.jpg/640px-Walking_tiger_female.jpg"
# url = "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcTUgaz9vFqcwV8XrisJxym7vZWp1hIagN1SDA&s"
# url = "https://www.littledayout.com/wp-content/uploads/lion-dance.jpg"

image = Image.open(requests.get(url, stream=True).raw)
display(image)

In [ ]:
from transformers import CLIPProcessor, CLIPModel

model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

labels = ["cat", "dog", "tiger", "train"]   #B
inputs = processor(text = labels, 
                   images = image, 
                   return_tensors = "pt", 
                   padding=True)

model.eval()

with torch.no_grad():
    outputs = model(**inputs)                             

logits_per_image = outputs.logits_per_image              
probs = logits_per_image.softmax(dim=1)

most_likely_index = torch.argmax(probs, dim=1).item()    
most_likely_object = labels[most_likely_index]

print(f"The most likely object is: {most_likely_object}")